In [ ]:
print("Snowflake is amazing...")

# Snowflake Warehouses & Roles

## Key Concept: Warehouses are Independent from Roles

Warehouses and roles are **separate objects** in Snowflake. However, a role needs the **USAGE** privilege granted on a warehouse to actually use it.

### How it works:

| Object | Purpose |
|--------|--------|
| **Warehouse** | Compute resources (exists on its own) |
| **Role** | A set of privileges (exists on its own) |

### Relationship:
- You **grant USAGE** on a warehouse to a role so that anyone with that role can run queries using that warehouse
- A single warehouse can be used by **many roles**
- A single role can have access to **many warehouses**

### Example:
```sql
GRANT USAGE ON WAREHOUSE ANALYST_WH TO ROLE DATA_ANALYST_ROLE;
```

> **Note:** The `DEFAULT_WAREHOUSE` setting on a user is just a convenience — it does not restrict which warehouses they can use.

# Snowflake Role Hierarchy

Roles in Snowflake follow a **hierarchy** — they are NOT independent. They are connected through a parent-child grant structure.

## Default Role Hierarchy (top to bottom):

```
ORGADMIN (organization-level, separate branch)

ACCOUNTADMIN
├── SYSADMIN (manages databases, warehouses, objects)
│   └── custom roles (should be granted to SYSADMIN)
└── SECURITYADMIN (manages grants and access)
    └── USERADMIN (manages users and roles)

PUBLIC (every user automatically has this role)
```

## Role Descriptions:

| Role | Purpose |
|------|--------|
| **ACCOUNTADMIN** | Top-level account role (inherits all from SYSADMIN + SECURITYADMIN) |
| **SYSADMIN** | Manages objects (databases, warehouses, schemas, tables) |
| **SECURITYADMIN** | Manages grants/privileges |
| **USERADMIN** | Manages users and roles |
| **PUBLIC** | Automatically granted to every user (Used for Shared objects which should be accessibe to everyone) |
| **ORGADMIN** | Manages organization-level operations (separate branch) |

## How Hierarchy Works:

- A parent role **inherits** all privileges of its child roles
- ACCOUNTADMIN can do everything SYSADMIN and SECURITYADMIN can do

## Best Practice:

Always grant custom roles to SYSADMIN so they stay within the hierarchy:

```sql
CREATE ROLE DATA_ANALYST_ROLE;
GRANT ROLE DATA_ANALYST_ROLE TO ROLE SYSADMIN;
```

> **Note:** Without this grant, custom roles become "orphaned" — accessible only by ACCOUNTADMIN directly.

# Assigning Privileges to a Role

Use the **GRANT** statement to assign privileges:

```sql
GRANT <privilege> ON <object_type> <object_name> TO ROLE <role_name>;
```

## Common Examples:

```sql
-- Warehouse access
GRANT USAGE ON WAREHOUSE my_wh TO ROLE my_role;

-- Database access
GRANT USAGE ON DATABASE my_db TO ROLE my_role;

-- Schema access
GRANT USAGE ON SCHEMA my_db.my_schema TO ROLE my_role;

-- Table access
GRANT SELECT ON TABLE my_db.my_schema.my_table TO ROLE my_role;

-- All tables in a schema
GRANT SELECT ON ALL TABLES IN SCHEMA my_db.my_schema TO ROLE my_role;

-- Future tables (auto-grant for tables created later)
GRANT SELECT ON FUTURE TABLES IN SCHEMA my_db.my_schema TO ROLE my_role;
```

## Key Privileges:

| Privilege | Used On | Purpose |
|-----------|---------|--------|
| USAGE | Warehouse, Database, Schema | Access the object |
| SELECT | Table, View | Read data |
| INSERT, UPDATE, DELETE | Table | Modify data |
| CREATE TABLE | Schema | Create new tables |
| CREATE VIEW | Schema | Create new views |
| OWNERSHIP | Any object | Full control |

## Typical Setup for a Read-Only Analyst Role:

```sql
GRANT USAGE ON WAREHOUSE analyst_wh TO ROLE analyst_role;
GRANT USAGE ON DATABASE analytics_db TO ROLE analyst_role;
GRANT USAGE ON SCHEMA analytics_db.public TO ROLE analyst_role;
GRANT SELECT ON ALL TABLES IN SCHEMA analytics_db.public TO ROLE analyst_role;
GRANT SELECT ON FUTURE TABLES IN SCHEMA analytics_db.public TO ROLE analyst_role;
```

> **Note:** You need USAGE on both the database AND schema before SELECT on tables will work — privileges don't cascade automatically.

# ALL TABLES vs FUTURE TABLES vs Specific Table Grants

| Grant Type | What it does |
|-----------|--------------|
| **ALL TABLES** | Grants privilege on all tables that **currently exist** in the schema |
| **FUTURE TABLES** | Grants privilege on tables that **will be created later** (doesn't affect existing ones) |
| **Specific table** | Grants privilege on **one specific table** only |

## Examples:

```sql
-- Only existing tables (new tables won't get this grant)
GRANT SELECT ON ALL TABLES IN SCHEMA my_db.my_schema TO ROLE my_role;

-- Only future tables (existing tables won't get this grant)
GRANT SELECT ON FUTURE TABLES IN SCHEMA my_db.my_schema TO ROLE my_role;

-- One specific table
GRANT SELECT ON TABLE my_db.my_schema.customers TO ROLE my_role;
```

## Best Practice — Use Both for Complete Coverage:

```sql
-- Cover existing tables
GRANT SELECT ON ALL TABLES IN SCHEMA my_db.my_schema TO ROLE my_role;
-- Cover any new tables created later
GRANT SELECT ON FUTURE TABLES IN SCHEMA my_db.my_schema TO ROLE my_role;
```

## Table-Specific Grants:

```sql
-- Grant SELECT on only one table
GRANT SELECT ON TABLE my_db.sales.orders TO ROLE my_role;

-- Grant different privileges on different tables
GRANT SELECT ON TABLE my_db.sales.orders TO ROLE my_role;
GRANT SELECT, INSERT ON TABLE my_db.sales.order_logs TO ROLE my_role;
```

> **Tip:** Use table-specific grants when you want fine-grained control — e.g., a role that can read orders but not customer PII tables.

# Snowflake Architecture

Snowflake uses a **3-layer architecture** where each layer is independent and can scale separately.

```
┌─────────────────────────────────────────────┐
│            CLOUD SERVICES                   │
│        (Brain of the system)                │
├─────────────────────────────────────────────┤
│          QUERY PROCESSING                   │
│        (Muscle of the system)               │
├─────────────────────────────────────────────┤
│              STORAGE                        │
│        (Hybrid Columnar Storage)            │
└─────────────────────────────────────────────┘
```

---

## Layer 1: Cloud Services (Brain of the System)

- Managing infrastructure
- Access control & security
- Query optimizer
- Metadata management
- Transaction management

---

## Layer 2: Query Processing (Muscle of the System)

- Performs **MPP (Massive Parallel Processing)**
- Uses virtual warehouses (independent compute clusters)
- Each warehouse operates independently — no resource contention
- Can scale up (bigger warehouse) or scale out (multi-cluster)

---

## Layer 3: Storage

- **Hybrid Columnar Storage** format
- Data is saved in **blobs** (cloud object storage: S3, Azure Blob, GCS)
- Data is compressed, encrypted, and stored in micro-partitions
- Storage is separate from compute — you pay for storage independently

---

## Key Takeaway:

The separation of these 3 layers means:
- **Storage** scales independently from **compute**
- Multiple warehouses can query the **same data** simultaneously
- You only pay for compute when queries are running

# Virtual Warehouses & Multi-Clustering

## What is a Warehouse?

A warehouse in Snowflake is a **cluster of compute resources** (CPU, memory, temp storage) that executes queries. It does NOT store data — it only provides compute.

**Key properties:**
- **Sizes:** XS, S, M, L, XL, 2XL, 3XL, 4XL, 5XL, 6XL (each size doubles the compute from the previous)
- **Auto-suspend:** Automatically shuts down after idle time (saves credits)
- **Auto-resume:** Automatically starts when a query arrives
- **Independent:** Multiple warehouses can query the same data without interfering with each other

---

## Multi-Cluster Warehouses

When a single warehouse gets overloaded (too many concurrent queries), Snowflake can **scale out** by adding additional clusters of the same size.

```
Standard Warehouse:          Multi-Cluster Warehouse:
┌──────────┐                 ┌──────────┐ ┌──────────┐ ┌──────────┐
│ Cluster  │                 │ Cluster 1│ │ Cluster 2│ │ Cluster 3│
│  (1x)    │                 │          │ │  (auto)  │ │  (auto)  │
└──────────┘                 └──────────┘ └──────────┘ └──────────┘
```

### Two Scaling Policies:

| Policy | Behavior |
|--------|----------|
| **Standard** | Adds clusters when queries queue up; removes when load decreases |
| **Economy** | Only adds clusters when the system estimates enough load to keep them busy for 6+ minutes |

### Configuration:

```sql
ALTER WAREHOUSE my_wh SET
  MIN_CLUSTER_COUNT = 1
  MAX_CLUSTER_COUNT = 3
  SCALING_POLICY = 'STANDARD';
```

---

## Scale Up vs Scale Out

| Approach | Solves | How |
|----------|--------|-----|
| **Scale Up** (L → XL) | Slow individual queries | More compute per query |
| **Scale Out** (multi-cluster) | Query queuing / concurrency | More parallel clusters handling different queries |

> **Note:** Multi-cluster is only available in Snowflake Enterprise Edition and above.

# Scaling Up a Warehouse

Use **ALTER WAREHOUSE** to change the size (scale up or down):

```sql
-- Scale up
ALTER WAREHOUSE my_wh SET WAREHOUSE_SIZE = 'LARGE';

-- Scale back down
ALTER WAREHOUSE my_wh SET WAREHOUSE_SIZE = 'XSMALL';
```

## Available Sizes:

`XSMALL` → `SMALL` → `MEDIUM` → `LARGE` → `XLARGE` → `2XLARGE` → `3XLARGE` → `4XLARGE` → `5XLARGE` → `6XLARGE`

(Each size doubles the compute from the previous)

## Behavior:

- Currently running queries finish on the **old size**
- New queries use the **new size**
- Change takes effect almost immediately

> **Note:** There is no automatic scale-up — you must manually change the size (or automate via a task/procedure). Multi-cluster auto-scaling only handles scale-out (adding more clusters of the same size).

# Scale Up vs Scale Out | Single-Cluster vs Multi-Cluster

## Scale Up (Upscale)

- Makes individual queries run **faster** by increasing compute power
- Does **not** provide parallelism — still one cluster handling queries
- Use when a single query is slow or complex

## Scale Out

- Spins up **new clusters** when multiple queries are queued for execution
- Provides **parallelism** — multiple queries run concurrently across different clusters
- Use when many users/queries are waiting in line

---

## Single-Cluster Warehouse

- A warehouse with a **single cluster**
- Can execute **multiple queries concurrently** (not just one), but has a concurrency limit
- When that limit is exceeded, additional queries **queue**

> **Correction:** A single-cluster warehouse CAN run multiple queries at the same time (up to its concurrency limit, typically 8). It's not limited to one query at a time — it just can't add more clusters when overloaded.

## Multi-Cluster Warehouse

- Allows **additional clusters** to spin up automatically when queries queue
- Each cluster handles different queries — true horizontal scaling
- Ideal for high-concurrency workloads

---

## Code Examples:

```sql
-- Single-cluster warehouse
CREATE WAREHOUSE IF NOT EXISTS ANALYST_WH
  WAREHOUSE_SIZE = 'XSMALL'
  AUTO_SUSPEND = 300        -- in seconds
  AUTO_RESUME = TRUE
  COMMENT = 'THIS IS SINGLE CLUSTER WAREHOUSE'
  SCALING_POLICY = 'ECONOMY'
  INITIALLY_SUSPENDED = TRUE;

-- Multi-cluster warehouse
CREATE WAREHOUSE IF NOT EXISTS ANALYST_M_WH
  WAREHOUSE_SIZE = 'XSMALL'
  MIN_CLUSTER_COUNT = 1
  MAX_CLUSTER_COUNT = 3
  SCALING_POLICY = 'STANDARD'
  AUTO_SUSPEND = 300        -- in seconds
  AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE;
```

> **Note:** The key difference in SQL is `MIN_CLUSTER_COUNT` and `MAX_CLUSTER_COUNT`. Without these, you get a single-cluster warehouse by default.

# Warehouse Concurrency Limit

The concurrency limit is **not set during warehouse creation** — it's a separate parameter you configure after:

```sql
ALTER WAREHOUSE ANALYST_WH SET MAX_CONCURRENCY_LEVEL = 8;
```

## Key Facts:

| Property | Value |
|----------|-------|
| Default | **8** concurrent queries per cluster |
| Range | 1 to 64 |
| Scope | Per cluster |

- A multi-cluster warehouse with 3 clusters can handle up to **3 × 8 = 24** concurrent queries
- When the limit is reached, additional queries go into a **queue**

## Check Current Setting:

```sql
SHOW PARAMETERS LIKE 'MAX_CONCURRENCY_LEVEL' IN WAREHOUSE ANALYST_WH;
```

## When to Adjust:

- **Lower it** → gives each query more resources (useful for heavy/complex queries)
- **Raise it** → allows more concurrent queries but each gets fewer resources

> **Note:** In most cases, the default of 8 works well.